# Confidence-Gated Recovery for KV-Cache Compression

**Base paper:** SnapKV (Li et al., NeurIPS 2024, arXiv:2404.14469).
SnapKV keeps the context tokens that the last 32 prompt tokens attend to and deletes the rest for good.

**What this notebook adds:** the deleted tokens are kept in a backup in CPU RAM. During generation we watch
the model's next-token confidence. When it drops below a threshold `tau`, we re-score the backup against the
query the model is using at that moment and swap the most relevant tokens back in. The number of context
tokens on the GPU never changes.

**How to run**
1. `Runtime -> Change runtime type -> T4 GPU`.
2. `Runtime -> Run all`. Allow the Google Drive prompt (results are saved there after every example).
3. If Colab disconnects, just `Run all` again. Finished work is skipped.

The last section builds every table and figure for the paper and zips them.

## 1. Settings

In [ ]:
MODEL_ID   = "Qwen/Qwen2.5-1.5B-Instruct"
DATASETS   = ["needle", "hotpotqa", "2wikimqa", "qasper"]
N_EXAMPLES = 30                     # per dataset. Lower this first if you are short on time.
MAX_LEN    = 6000                   # prompt is cut in the middle to this many tokens (LongBench style)
RATIOS     = [0.02, 0.05, 0.10]     # fraction of the context kept on the GPU
TAUS       = [0.5, 0.7, 0.9]        # confidence thresholds for recovery
HEAD_TAU   = 0.7                    # the threshold reported as "ours" in the main table (fixed before running)
WINDOW     = 32                     # SnapKV observation window
USE_DRIVE  = True                   # save results to Google Drive so a disconnect loses nothing
RUN_NAME   = "cgr_qwen1.5b_fp32"    # new name: the earlier fp16 run produced NaN and must not be resumed

## 2. Environment

In [ ]:
import os, sys, gc, json, time, random, platform
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
import torch, transformers
print("python", platform.python_version(), "| torch", torch.__version__, "| transformers", transformers.__version__)
assert torch.cuda.is_available(), "No GPU. Runtime -> Change runtime type -> T4 GPU"
print(torch.cuda.get_device_name(0), f"{torch.cuda.get_device_properties(0).total_memory / 2**30:.1f} GB")

BASE = next(d for d in ("/content", "/kaggle/working", ".") if os.path.isdir(d))   # Colab, Kaggle, anywhere else
OUT_DIR = f"{BASE}/{RUN_NAME}"
if USE_DRIVE and BASE == "/content":
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        OUT_DIR = f"/content/drive/MyDrive/{RUN_NAME}"
    except Exception as e:
        print("Drive not mounted, saving to /content instead:", e)
os.makedirs(OUT_DIR, exist_ok=True)
RESULTS = f"{OUT_DIR}/results.jsonl"
print("results ->", RESULTS)

## 3. The method

One file holds everything: the shared prefill, the four baselines, our recovery step, the data loaders,
the metric and the crash-safe runner. It is written to disk and imported so the same code can be unit-tested
outside Colab.

In [ ]:
%%writefile cgr.py
"""Confidence-Gated Recovery (CGR) for KV-cache compression.

Base method: SnapKV (Li et al., NeurIPS 2024). SnapKV scores context tokens by
the attention they receive from the last `window` prompt tokens, keeps the top
ones and deletes the rest for good.

This module adds one thing on top: the deleted tokens are kept in a backup, and
while the model generates we watch its next-token confidence. When confidence
drops below a threshold, we re-score the backup against the query the model is
using right now and swap the most relevant tokens back in. The number of
context slots on the GPU never changes.

Everything (baselines and our method) runs through the same code path, so the
comparison is like for like: same prompt, same window, same budget.

Works with Hugging Face Llama/Qwen2-style models (RoPE, grouped-query
attention) loaded with attn_implementation="eager".
"""

from __future__ import annotations

import gc
import io
import json
import math
import os
import random
import re
import string
import time
import zipfile
from collections import Counter
from dataclasses import dataclass, field

import torch
import torch.nn.functional as F
from transformers import DynamicCache

N_SINK = 4  # first tokens are never evicted (attention sinks)


# --------------------------------------------------------------------------
# Cache helpers. We keep the cache as a plain list of (K, V) per layer and only
# wrap it in a DynamicCache for the duration of one forward call. This avoids
# depending on DynamicCache internals, which changed between versions.
# --------------------------------------------------------------------------

def cache_to_list(cache):
    if hasattr(cache, "layers"):  # transformers >= 4.54
        return [(layer.keys, layer.values) for layer in cache.layers]
    if hasattr(cache, "key_cache"):  # transformers <= 4.53
        return list(zip(cache.key_cache, cache.value_cache))
    return [(layer[0], layer[1]) for layer in cache]


def list_to_cache(kvs):
    cache = DynamicCache()
    for i, (k, v) in enumerate(kvs):
        cache.update(k, v, i)
    return cache


def forward(model, ids, pos, kvs, output_attentions=False):
    """One forward call. `ids` and `pos` are 1-D tensors of the new tokens and
    their ORIGINAL positions. Returns (logits of last token, new kv list, attentions)."""
    out = model(
        input_ids=ids[None],
        position_ids=pos[None],
        past_key_values=list_to_cache(kvs) if kvs else DynamicCache(),
        use_cache=True,
        output_attentions=output_attentions,
    )
    return out.logits[0, -1].float(), cache_to_list(out.past_key_values), out.attentions


def kv_bytes(kvs):
    """Exact KV-cache size in bytes, counted from the tensors themselves."""
    return sum(k.numel() * k.element_size() + v.numel() * v.element_size() for k, v in kvs)


# --------------------------------------------------------------------------
# Query capture. A forward hook on every q_proj stores the query vectors of the
# most recent forward call; recovery uses the last token's query.
# --------------------------------------------------------------------------

class QueryTap:
    def __init__(self, model):
        self.model = model
        cfg = model.config
        self.n_heads = cfg.num_attention_heads
        self.n_kv = getattr(cfg, "num_key_value_heads", self.n_heads)
        self.group = self.n_heads // self.n_kv
        self.head_dim = getattr(cfg, "head_dim", None) or cfg.hidden_size // self.n_heads
        self.layers = model.model.layers
        self.q = [None] * len(self.layers)
        self.handles = [
            layer.self_attn.q_proj.register_forward_hook(self._hook(i))
            for i, layer in enumerate(self.layers)
        ]
        # RoPE frequencies: read them from the model itself. Newer transformers versions
        # moved `rope_theta` inside `config.rope_parameters`, so reading the config
        # attribute silently fell back to a wrong default there.
        rot = getattr(model.model, "rotary_emb", None)
        if rot is not None and getattr(rot, "inv_freq", None) is not None:
            self.inv_freq = rot.inv_freq.detach().float().cpu().reshape(-1)
        else:
            theta = getattr(cfg, "rope_theta", None) or (getattr(cfg, "rope_parameters", None) or {}).get("rope_theta")
            assert theta, "cannot find the RoPE base of this model"
            d = self.head_dim
            self.inv_freq = 1.0 / (theta ** (torch.arange(0, d, 2, dtype=torch.float32) / d))
        assert self.inv_freq.numel() == self.head_dim // 2

    def _hook(self, i):
        def fn(module, inputs, output):
            self.q[i] = output[0].detach()  # every token of the call, (T, n_heads * head_dim)
        return fn

    def rotated_queries(self, position: int):
        """Last-token queries of every layer with RoPE applied -> list of (n_heads, d) fp32."""
        freqs = position * self.inv_freq
        emb = torch.cat([freqs, freqs])
        cos, sin = emb.cos(), emb.sin()
        out = []
        for q in self.q:
            q = q[-1].float().view(self.n_heads, self.head_dim)
            cos_d, sin_d = cos.to(q.device), sin.to(q.device)
            half = self.head_dim // 2
            rot = torch.cat([-q[:, half:], q[:, :half]], dim=-1)
            out.append(q * cos_d + rot * sin_d)
        return out

    def rotated_chunk(self, layer: int, positions):
        """All queries of the last forward call for one layer, RoPE applied -> (T, n_heads, d) fp32."""
        q = self.q[layer].float().view(-1, self.n_heads, self.head_dim)
        freqs = positions.float()[:, None] * self.inv_freq.to(q.device)[None, :]
        emb = torch.cat([freqs, freqs], -1)[:, None, :]            # (T, 1, d)
        half = self.head_dim // 2
        rot = torch.cat([-q[..., half:], q[..., :half]], dim=-1)
        return q * emb.cos() + rot * emb.sin()

    def remove(self):
        for h in self.handles:
            h.remove()


# --------------------------------------------------------------------------
# Prefill: run the prompt once, uncompressed, and collect everything every
# method needs. Done once per example and shared by all methods and budgets.
# --------------------------------------------------------------------------

@dataclass
class Prefill:
    kvs: list            # full cache, per layer (K, V), shape (1, n_kv, n, d)
    n: int               # prompt length in tokens
    n_ctx: int           # compressible prefix length (= n - window)
    first_logits: torch.Tensor
    win_score: torch.Tensor   # (L, n_kv, n_ctx) mean attention from the window queries
    acc_score: torch.Tensor   # (L, n_kv, n_ctx) attention accumulated over ALL queries (H2O)
    seconds: float


@torch.no_grad()
def prefill(model, tap: QueryTap, input_ids, window=32, chunk=256):
    t0 = time.time()
    n = input_ids.numel()
    assert n > window + N_SINK + 8, "prompt too short to compress"
    n_ctx = n - window
    dev = input_ids.device
    L, n_kv, G = len(tap.layers), tap.n_kv, tap.group
    acc = torch.zeros(L, n_kv, n, device=dev)
    win = torch.zeros(L, n_kv, n_ctx, device=dev)
    kvs, logits = [], None
    bounds = list(range(0, n_ctx, chunk)) + [n_ctx, n]
    for a, b in zip(bounds[:-1], bounds[1:]):
        if a == b:
            continue
        pos = torch.arange(a, b, device=dev)
        logits, kvs, _ = forward(model, input_ids[a:b], pos, kvs)
        # The attention weights are recomputed here from the hooked queries and the
        # cached keys, one layer at a time. Asking the model for output_attentions
        # keeps every layer's (heads x chunk x prompt) matrix alive at once, which
        # does not fit on a 16 GB GPU for long prompts.
        future = torch.arange(b, device=dev)[None, :] > pos[:, None]          # (T, b) causal mask
        for l in range(L):
            q = tap.rotated_chunk(l, pos).view(b - a, n_kv, G, tap.head_dim)
            keys = kvs[l][0][0, :, :b].float()                                # (n_kv, b, d)
            att = torch.einsum("qkgd,ktd->kgqt", q, keys) / math.sqrt(tap.head_dim)
            att = torch.softmax(att.masked_fill_(future, float("-inf")), -1).mean(1)   # (n_kv, T, b)
            acc[l, :, :b] += att.sum(1)
            if a == n_ctx:  # this chunk is exactly the observation window
                win[l] = att[:, :, :n_ctx].mean(1)
            del q, keys, att
    # fp16 can overflow on long prompts (seen with Qwen2.5-1.5B on a T4: every logit NaN,
    # every answer "!!!!"). Stop here rather than write garbage results.
    if not torch.isfinite(logits).all():
        raise FloatingPointError("the model produced NaN/inf logits for this prompt; load it in float32")
    return Prefill(kvs, n, n_ctx, logits, win, acc[:, :, :n_ctx].contiguous(), time.time() - t0)


# --------------------------------------------------------------------------
# Token selection. Every method returns sorted indices (L, n_kv, B) into the
# compressible prefix. Sink tokens are always kept.
# --------------------------------------------------------------------------

def pool(score, kernel=7):
    """SnapKV's local pooling: keeps the neighbours of a high-scoring token."""
    L, H, T = score.shape
    return F.avg_pool1d(score.view(L * H, 1, T), kernel, stride=1, padding=kernel // 2).view(L, H, T)


def top_b(score, B):
    score = score.clone()
    score[..., :N_SINK] = float("inf")
    return score.topk(B, dim=-1).indices.sort(dim=-1).values


def budget_tokens(n_ctx, ratio):
    return max(N_SINK + 4, min(n_ctx, round(ratio * n_ctx)))


def select(method, pf: Prefill, B, seed=0, kernel=7):
    L, H, T = pf.win_score.shape
    dev = pf.win_score.device
    position = torch.arange(T, device=dev, dtype=torch.float32).expand(L, H, T)
    if method == "full":
        return torch.arange(T, device=dev).expand(L, H, T)
    if method == "random":
        g = torch.Generator().manual_seed(seed)
        return top_b(torch.rand(L, H, T, generator=g).to(dev), B)
    if method == "streaming":  # StreamingLLM: sinks + most recent tokens
        return top_b(position, B)
    if method == "h2o":  # H2O: half heavy hitters (accumulated attention), half recent
        score = pf.acc_score.clone()
        n_recent = B // 2
        score[..., T - n_recent:] = float("inf")
        return top_b(score, B)
    if method == "snapkv":
        return top_b(pool(pf.win_score, kernel), B)
    raise ValueError(method)


def gather_kv(kvs, idx, n_ctx, device=None):
    """Build the compressed cache: selected prefix tokens + everything after the prefix."""
    out = []
    for (k, v), ix in zip(kvs, idx):
        ix = ix.to(k.device)[None, :, :, None].expand(1, -1, -1, k.shape[-1])
        ck = torch.cat([k[:, :, :n_ctx].gather(2, ix).to(device or k.device), k[:, :, n_ctx:].to(device or k.device)], 2)
        cv = torch.cat([v[:, :, :n_ctx].gather(2, ix).to(device or v.device), v[:, :, n_ctx:].to(device or v.device)], 2)
        out.append((ck, cv))
    return out


# --------------------------------------------------------------------------
# Generation with optional confidence-gated recovery.
# --------------------------------------------------------------------------

@dataclass
class GenConfig:
    method: str = "snapkv"        # full | random | streaming | h2o | snapkv
    ratio: float = 0.1            # fraction of the compressible prefix that is kept
    trigger: str = "never"        # never | conf | always | random
    tau: float = 0.5              # conf: recover when max next-token prob < tau
    p_random: float = 0.0         # random: per-step trigger probability
    cooldown: int = 2             # steps to wait after a recovery
    kernel: int = 7
    max_new_tokens: int = 32
    seed: int = 0

    def name(self):
        if self.method == "full":
            return "full"
        tag = {"never": "", "conf": f"+cgr(tau={self.tau})", "always": "+always", "random": "+randtrig"}[self.trigger]
        return f"{self.method}{tag}"


@dataclass
class GenResult:
    tokens: list
    conf: list                    # confidence at each step, before any recovery
    trig_steps: list = field(default_factory=list)
    conf_after: list = field(default_factory=list)   # confidence after recovery, per trigger
    flipped: int = 0              # triggers where recovery changed the chosen token
    swapped: float = 0.0          # mean fraction of slots replaced per recovery
    kept_tokens: int = 0
    gpu_kv_bytes: int = 0
    backup_bytes: int = 0
    gen_seconds: float = 0.0
    rec_seconds: float = 0.0


@torch.no_grad()
def recover(tap: QueryTap, backup, tail_kvs, keep_score, position, B, kernel=7):
    """Re-score every backed-up prefix token against the current query and
    return (new indices, updated scores).

    backup:     per layer (K, V) of the full prefix, held off the GPU
    tail_kvs:   per layer (K, V) of the tokens after the prefix (window + generated)
    keep_score: (L, n_kv, n_ctx) the scores the current selection was made from

    For each layer we compute the attention the current query WOULD put on every
    prefix token if the whole prefix were still in the cache. Only one layer's
    keys are on the GPU at a time. The new score is max(old, new), so a token
    that mattered for the prompt is not thrown out by one unrelated query.
    """
    qs = tap.rotated_queries(position)
    n_kv, G, d = tap.n_kv, tap.group, tap.head_dim
    new = torch.empty_like(keep_score)
    for l, q in enumerate(qs):
        dev = q.device
        k_ctx = backup[l][0][0].to(dev).float()            # (n_kv, n_ctx, d)
        k_tail = tail_kvs[l][0][0].float()                 # (n_kv, t, d)
        keys = torch.cat([k_ctx, k_tail], 1)               # (n_kv, n_ctx + t, d)
        q = q.view(n_kv, G, d)
        att = torch.softmax(torch.einsum("kgd,ktd->kgt", q, keys) / math.sqrt(d), -1)
        new[l] = att[:, :, : k_ctx.shape[1]].mean(1).to(new.device)
        del k_ctx, keys
    score = torch.maximum(keep_score, pool(new, kernel))
    return top_b(score, B), score


@torch.no_grad()
def generate(model, tap: QueryTap, pf: Prefill, cfg: GenConfig, eos_ids, backup=None):
    """Greedy decoding from a compressed cache.

    As in SnapKV, the first token comes from the uncompressed prefill pass and
    compression applies from the second token on. This is identical for all methods.
    `backup` is the prefix KV on the backup device (CPU on a GPU run); it is
    only needed when cfg.trigger != "never".
    """
    dev = pf.first_logits.device
    n_ctx = pf.n_ctx
    B = n_ctx if cfg.method == "full" else budget_tokens(n_ctx, cfg.ratio)
    idx = select(cfg.method, pf, B, cfg.seed, cfg.kernel)
    kvs = gather_kv(pf.kvs, idx, n_ctx)
    res = GenResult(tokens=[], conf=[], kept_tokens=B + (pf.n - n_ctx))
    res.gpu_kv_bytes = kv_bytes(kvs)
    use_rec = cfg.trigger != "never" and cfg.method != "full"
    if use_rec:
        assert backup is not None
        res.backup_bytes = kv_bytes(backup)
        keep_score = pool(pf.win_score, cfg.kernel) if cfg.method == "snapkv" else torch.zeros_like(pf.win_score)
    rng = random.Random(cfg.seed)
    t0 = time.time()

    probs = torch.softmax(pf.first_logits, -1)
    res.conf.append(probs.max().item())
    tok = int(probs.argmax())
    res.tokens.append(tok)
    wait = 0
    for step in range(1, cfg.max_new_tokens):
        if tok in eos_ids:
            break
        pos = pf.n + step - 1
        ids_t = torch.tensor([tok], device=dev)
        pos_t = torch.tensor([pos], device=dev)
        logits, new_kvs, _ = forward(model, ids_t, pos_t, kvs)
        probs = torch.softmax(logits, -1)
        conf = probs.max().item()
        res.conf.append(conf)

        fire = False
        if use_rec and wait == 0:
            if cfg.trigger == "always":
                fire = True
            elif cfg.trigger == "conf":
                fire = conf < cfg.tau
            elif cfg.trigger == "random":
                fire = rng.random() < cfg.p_random
        wait = max(0, wait - 1)

        if fire:
            t1 = time.time()
            tail = [(k[:, :, B:], v[:, :, B:]) for k, v in kvs]   # before this step's token
            new_idx, keep_score = recover(tap, backup, tail, keep_score, pos, B, cfg.kernel)
            changed = 1.0 - (new_idx[..., None] == idx[..., None, :]).any(-1).float().mean().item()
            idx = new_idx
            ctx = gather_kv(backup, idx, n_ctx, device=dev)        # prefix slots only (backup has no tail)
            kvs = [(torch.cat([ck, tk], 2), torch.cat([cv, tv], 2)) for (ck, cv), (tk, tv) in zip(ctx, tail)]
            logits, new_kvs, _ = forward(model, ids_t, pos_t, kvs)  # decode the same step again
            probs2 = torch.softmax(logits, -1)
            res.trig_steps.append(step)
            res.conf_after.append(probs2.max().item())
            res.flipped += int(probs2.argmax() != probs.argmax())
            res.swapped += changed
            probs = probs2
            wait = cfg.cooldown
            res.rec_seconds += time.time() - t1
        kvs = new_kvs
        tok = int(probs.argmax())
        res.tokens.append(tok)

    assert all(k.shape[2] == B + (pf.n - n_ctx) + len(res.tokens) - 1 for k, _ in kvs), "cache size drifted"
    if res.trig_steps:
        res.swapped /= len(res.trig_steps)
    res.gen_seconds = time.time() - t0
    return res


def make_backup(pf: Prefill, device="cpu"):
    """Copy of the compressible prefix, kept off the GPU."""
    return [(k[:, :, : pf.n_ctx].to(device), v[:, :, : pf.n_ctx].to(device)) for k, v in pf.kvs]


# --------------------------------------------------------------------------
# Data: LongBench QA sets and a synthetic two-needle test.
# --------------------------------------------------------------------------

QA_PROMPT = (
    "Answer the question based on the given passages. Only give me the answer and do not output any other words."
    "\n\nThe following are given passages.\n{context}\n\nAnswer the question based on the given passages. "
    "Only give me the answer and do not output any other words.\n\nQuestion: {input}\nAnswer:"
)
PROMPTS = {
    "hotpotqa": QA_PROMPT,
    "2wikimqa": QA_PROMPT,
    "musique": QA_PROMPT,
    "qasper": (
        "You are given a scientific article and a question. Answer the question as concisely as you can, using a "
        "single phrase or sentence if possible. If the question cannot be answered based on the information in the "
        "article, write \"unanswerable\". If the question is a yes/no question, answer \"yes\", \"no\", or "
        "\"unanswerable\". Do not provide any explanation.\n\nArticle: {context}\n\n Answer the question based on "
        "the above article as concisely as you can, using a single phrase or sentence if possible. If the question "
        "cannot be answered based on the information in the article, write \"unanswerable\". If the question is a "
        "yes/no question, answer \"yes\", \"no\", or \"unanswerable\". Do not provide any explanation.\n\n"
        "Question: {input}\n\nAnswer:"
    ),
    "needle": (
        "Some special magic numbers are hidden in the following text. Read it carefully.\n\n{context}\n\n"
        "Answer with the two numbers only. What are the special magic numbers for {input}?\nAnswer:"
    ),
}
MAX_NEW = {"hotpotqa": 32, "2wikimqa": 32, "musique": 32, "qasper": 128, "needle": 24}


def load_longbench(name):
    from huggingface_hub import hf_hub_download
    path = hf_hub_download("THUDM/LongBench", "data.zip", repo_type="dataset")
    with zipfile.ZipFile(path) as z:
        with z.open(f"data/{name}.jsonl") as f:
            return [json.loads(line) for line in io.TextIOWrapper(f, encoding="utf-8")]


CITIES = ["Lisbon", "Nairobi", "Osaka", "Quito", "Tallinn", "Hanoi", "Bogota", "Tunis", "Perth", "Oslo",
          "Lagos", "Minsk", "Havana", "Dhaka", "Zagreb", "Accra", "Kyoto", "Lima", "Riga", "Doha"]


def make_needles(n_examples, source_texts, tok, ctx_tokens, seed=0):
    """Two 'magic number' sentences hidden at random depths in real text.
    The text is cut to `ctx_tokens` BEFORE the needles go in, so truncation can never remove them."""
    rng = random.Random(seed)
    out = []
    for i in range(n_examples):
        text = source_texts[i % len(source_texts)]
        text = tok.decode(tok(text, add_special_tokens=False).input_ids[:ctx_tokens], skip_special_tokens=True)
        paras = [p for p in text.split("\n") if p.strip()]
        a, b = rng.sample(CITIES, 2)
        na, nb = rng.randint(1000000, 9999999), rng.randint(1000000, 9999999)
        for city, num in ((a, na), (b, nb)):
            paras.insert(rng.randint(1, max(1, len(paras) - 1)), f"The special magic number for {city} is {num}.")
        out.append({"context": "\n".join(paras), "input": f"{a} and {b}", "answers": [str(na), str(nb)]})
    return out


def build_prompt(tok, name, ex, max_len):
    """LongBench-style: fill the template, cut the middle if too long, wrap in the chat template."""
    prompt = PROMPTS[name].format(context=ex["context"], input=ex["input"])
    ids = tok(prompt, add_special_tokens=False).input_ids
    if len(ids) > max_len:
        half = max_len // 2
        prompt = tok.decode(ids[:half], skip_special_tokens=True) + tok.decode(ids[-half:], skip_special_tokens=True)
    text = tok.apply_chat_template([{"role": "user", "content": prompt}], tokenize=False, add_generation_prompt=True)
    return tok(text, add_special_tokens=False, return_tensors="pt").input_ids[0]


# --------------------------------------------------------------------------
# Metrics (LongBench's QA F1).
# --------------------------------------------------------------------------

def normalize_answer(s):
    s = "".join(ch for ch in s.lower() if ch not in set(string.punctuation))
    s = re.sub(r"\b(a|an|the)\b", " ", s)
    return " ".join(s.split())


def qa_f1(pred, answers):
    best = 0.0
    p = normalize_answer(pred).split()
    for a in answers:
        g = normalize_answer(a).split()
        same = sum((Counter(p) & Counter(g)).values())
        if same == 0:
            continue
        prec, rec = same / len(p), same / len(g)
        best = max(best, 2 * prec * rec / (prec + rec))
    return best


def score_output(name, pred, answers):
    if name == "needle":
        return sum(a in pred for a in answers) / len(answers)
    return qa_f1(pred, answers)


# --------------------------------------------------------------------------
# Experiment runner. Appends one JSON line per (example, config) and skips
# anything already in the file, so a dead Colab session loses nothing.
# --------------------------------------------------------------------------

def config_grid(ratios, taus):
    grid = [GenConfig(method="full")]
    for r in ratios:
        for m in ("random", "streaming", "h2o", "snapkv"):
            grid.append(GenConfig(method=m, ratio=r))
        for tau in taus:
            grid.append(GenConfig(method="snapkv", ratio=r, trigger="conf", tau=tau))
        grid.append(GenConfig(method="snapkv", ratio=r, trigger="always", cooldown=0))
        grid.append(GenConfig(method="snapkv", ratio=r, trigger="random"))  # p set per example
    return grid


def run_key(dataset, i, cfg):
    return f"{dataset}|{i}|{cfg.name()}|{cfg.ratio if cfg.method != 'full' else 1.0}"


def load_done(path):
    done = set()
    if os.path.exists(path):
        with open(path, encoding="utf-8") as f:
            for line in f:
                try:
                    done.add(json.loads(line)["key"])
                except Exception:
                    pass  # half-written last line from a killed session
    return done


def run_dataset(model, tok, tap, dataset, examples, out_path, ratios, taus, max_len=6000,
                window=32, ref_tau=0.7, backup_device="cpu", log_every=5):
    """Run every config on every example. `ref_tau` is the threshold whose
    trigger rate the random-trigger ablation copies, example by example."""
    done = load_done(out_path)
    grid = config_grid(ratios, taus)
    eos = set(model.generation_config.eos_token_id if isinstance(model.generation_config.eos_token_id, list)
              else [model.generation_config.eos_token_id]) | {tok.eos_token_id}
    dev = next(model.parameters()).device
    t_start = time.time()
    for i, ex in enumerate(examples):
        todo = [c for c in grid if run_key(dataset, i, c) not in done]
        if not todo:
            continue
        ids = build_prompt(tok, dataset, ex, max_len).to(dev)
        pf = backup = None
        gc.collect()
        if dev.type == "cuda":
            torch.cuda.empty_cache()
        pf = prefill(model, tap, ids, window=window)
        backup = make_backup(pf, backup_device)
        ref_rate = {}  # ratio -> trigger rate of the reference CGR run on this example
        # reference CGR runs first so the random-trigger ablation can copy their rate
        todo.sort(key=lambda c: (c.trigger == "random", not (c.trigger == "conf" and c.tau == ref_tau)))
        with open(out_path, "a", encoding="utf-8") as f:
            for cfg in todo:
                cfg.max_new_tokens = MAX_NEW[dataset]
                cfg.seed = i
                if cfg.trigger == "random":
                    if cfg.ratio not in ref_rate:
                        continue  # reference run was done in an earlier session; rate unknown -> skip
                    cfg.p_random = ref_rate[cfg.ratio]
                r = generate(model, tap, pf, cfg, eos, backup if cfg.trigger != "never" else None)
                steps = max(1, len(r.tokens) - 1)
                if cfg.trigger == "conf" and cfg.tau == ref_tau:
                    ref_rate[cfg.ratio] = len(r.trig_steps) / steps
                text = tok.decode(r.tokens, skip_special_tokens=True).strip()
                row = {
                    "key": run_key(dataset, i, cfg), "dataset": dataset, "idx": i, "name": cfg.name(),
                    "method": cfg.method, "trigger": cfg.trigger, "tau": cfg.tau if cfg.trigger == "conf" else None,
                    "ratio": cfg.ratio if cfg.method != "full" else 1.0,
                    "score": score_output(dataset, text, ex["answers"]), "pred": text,
                    "n_prompt": pf.n, "kept_tokens": r.kept_tokens, "steps": steps,
                    "n_trig": len(r.trig_steps), "trig_steps": r.trig_steps, "flipped": r.flipped,
                    "swapped": r.swapped, "conf": [round(c, 4) for c in r.conf],
                    "conf_after": [round(c, 4) for c in r.conf_after],
                    "gpu_kv_bytes": r.gpu_kv_bytes, "full_kv_bytes": kv_bytes(pf.kvs),
                    "backup_bytes": r.backup_bytes, "gen_seconds": r.gen_seconds,
                    "rec_seconds": r.rec_seconds, "prefill_seconds": pf.seconds,
                }
                f.write(json.dumps(row) + "\n")
                f.flush()
        del pf, backup
        if (i + 1) % log_every == 0 or i + 1 == len(examples):
            mem = f", peak GPU {torch.cuda.max_memory_allocated() / 2**30:.1f} GB" if dev.type == "cuda" else ""
            print(f"[{dataset}] {i + 1}/{len(examples)} examples, {time.time() - t_start:.0f}s{mem}", flush=True)

## 4. Load the model

In [ ]:
import importlib, cgr
importlib.reload(cgr)
from transformers import AutoModelForCausalLM, AutoTokenizer

tok = AutoTokenizer.from_pretrained(MODEL_ID)
# float32, not float16: in fp16 this model returned NaN for every long prompt on the T4 (all answers "!!!!").
# The 1.5B model in fp32 needs about 6 GB, which fits. eager attention: sanity check 4 reads the attention weights.
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=torch.float32, attn_implementation="eager").cuda().eval()
tap = cgr.QueryTap(model)
EOS = set(model.generation_config.eos_token_id if isinstance(model.generation_config.eos_token_id, list)
          else [model.generation_config.eos_token_id]) | {tok.eos_token_id}
print("layers", len(tap.layers), "| query heads", tap.n_heads, "| KV heads", tap.n_kv, "| head dim", tap.head_dim)

## 5. Sanity checks

These must pass before any number is trusted.
1. Our decoding loop with the full cache gives exactly the same tokens as `model.generate`.
2. A 100% budget gives exactly the full-cache output, with and without recovery.
3. Recovery never changes the size of the GPU cache.
4. The attention we compute by hand for recovery matches the model's own attention.

In [ ]:
import math
story = ("The harbour town of Velmora was founded in 1820 by salt traders. Its lighthouse keeper, Ilse Brandt, "
         "kept a ledger of every ship. The special magic number for Lisbon is 4815162. In winter the fishing fleet "
         "stayed in port and the market sold dried cod. The council met on the first Monday of each month. ") * 8
msgs = [{"role": "user", "content": story + "\nWhat is the special magic number for Lisbon? Answer with the number only."}]
ids = tok(tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True),
          add_special_tokens=False, return_tensors="pt").input_ids[0].cuda()
pf = cgr.prefill(model, tap, ids, window=WINDOW)
backup = cgr.make_backup(pf)

with torch.no_grad():
    ref = model.generate(ids[None], max_new_tokens=16, do_sample=False)[0, ids.numel():].tolist()
full = cgr.generate(model, tap, pf, cgr.GenConfig(method="full", max_new_tokens=16), EOS)
assert full.tokens == ref, "decoding loop differs from model.generate"
print("1. identity with model.generate: OK ->", repr(tok.decode(full.tokens, skip_special_tokens=True)))

for trig in ("never", "always"):
    r = cgr.generate(model, tap, pf, cgr.GenConfig(method="snapkv", ratio=1.0, trigger=trig, cooldown=0, max_new_tokens=16), EOS, backup)
    assert r.tokens == full.tokens and r.gpu_kv_bytes == cgr.kv_bytes(pf.kvs)
print("2. 100% budget equals full cache: OK")

base = cgr.generate(model, tap, pf, cgr.GenConfig(method="snapkv", ratio=0.1, max_new_tokens=16), EOS)
rec = cgr.generate(model, tap, pf, cgr.GenConfig(method="snapkv", ratio=0.1, trigger="always", cooldown=0, max_new_tokens=16), EOS, backup)
assert rec.gpu_kv_bytes == base.gpu_kv_bytes and len(rec.trig_steps) == len(rec.tokens) - 1
print(f"3. GPU cache size unchanged by recovery: OK ({base.gpu_kv_bytes / 2**20:.2f} MB of {cgr.kv_bytes(pf.kvs) / 2**20:.2f} MB)")

nxt, pos = torch.tensor([full.tokens[0]]).cuda(), torch.tensor([pf.n]).cuda()
with torch.no_grad():
    _, _, attns = cgr.forward(model, nxt, pos, pf.kvs, output_attentions=True)
qs, worst = tap.rotated_queries(pf.n), 0.0
for l in range(len(qs)):
    q = qs[l].view(tap.n_kv, tap.group, tap.head_dim)
    mine = torch.softmax(torch.einsum("kgd,ktd->kgt", q, pf.kvs[l][0][0].float()) / math.sqrt(tap.head_dim), -1)
    theirs = attns[l][0, :, 0, :-1].float().view(tap.n_kv, tap.group, -1)
    worst = max(worst, (mine - theirs / theirs.sum(-1, keepdim=True)).abs().max().item())
assert worst < 2e-2, worst
print(f"4. hand-computed attention matches the model: OK (max abs diff {worst:.1e})")
del pf, backup, attns; gc.collect(); torch.cuda.empty_cache()

# 5. Memory: one worst-case prompt (MAX_LEN tokens) through the heaviest configuration, before the long run.
torch.cuda.reset_peak_memory_stats()
long_ids = torch.randint(1000, 20000, (MAX_LEN,), generator=torch.Generator().manual_seed(0)).cuda()
pf = cgr.prefill(model, tap, long_ids, window=WINDOW)
backup = cgr.make_backup(pf)
cgr.generate(model, tap, pf, cgr.GenConfig(method="full", max_new_tokens=8), EOS)
cgr.generate(model, tap, pf, cgr.GenConfig(method="snapkv", ratio=0.2, trigger="always", cooldown=0, max_new_tokens=8), EOS, backup)
peak = torch.cuda.max_memory_allocated() / 2**30
total = torch.cuda.get_device_properties(0).total_memory / 2**30
print(f"5. memory for a {MAX_LEN}-token prompt: peak {peak:.1f} GB of {total:.1f} GB, prefill {pf.seconds:.0f}s", "OK" if peak < 0.8 * total else "TOO HIGH")
assert peak < 0.8 * total, "lower MAX_LEN before running the experiments"
del pf, backup, long_ids; gc.collect(); torch.cuda.empty_cache()

## 6. Data

Three LongBench question-answering sets and a synthetic two-needle test (two "magic number" sentences hidden in
real text; the score is the fraction of the two numbers the model returns). Examples are picked with a fixed seed.

In [ ]:
data = {}
hp = cgr.load_longbench("hotpotqa")
random.Random(0).shuffle(hp)
for name in DATASETS:
    if name == "needle":   # haystacks come from hotpotqa passages that are NOT used in the hotpotqa evaluation
        data[name] = cgr.make_needles(N_EXAMPLES, [e["context"] for e in hp[N_EXAMPLES:2 * N_EXAMPLES]], tok, MAX_LEN - 300)
    elif name == "hotpotqa":
        data[name] = hp[:N_EXAMPLES]
    else:
        rows = cgr.load_longbench(name)
        random.Random(0).shuffle(rows)
        data[name] = rows[:N_EXAMPLES]
for name, rows in data.items():
    n_tok = [cgr.build_prompt(tok, name, ex, MAX_LEN).numel() for ex in rows[:10]]
    print(f"{name:10s} {len(rows)} examples, ~{sum(n_tok) // len(n_tok)} prompt tokens")

## 7. Run the experiments

Per example the prompt is processed once, uncompressed. Every method and budget then starts from that same
cache, so all methods see exactly the same prompt. Configurations per example:

- `full`: no compression
- `random`, `streaming` (StreamingLLM), `h2o` (H2O), `snapkv` (SnapKV): baselines
- `snapkv+cgr(tau=...)`: ours, recovery when confidence < tau
- `snapkv+always`: recovery on every step (upper bound, the cost existing retrieval methods pay)
- `snapkv+randtrig`: recovery at random steps, as often as ours fired on that example (is confidence the right signal?)

In [ ]:
# Pilot: two real examples, printed, before the long run. The full-cache answers must be sensible text.
# If they are empty or look like "!!!!", STOP and do not run the next cell.
for name in DATASETS[:2]:
    ex = data[name][0]
    pf = cgr.prefill(model, tap, cgr.build_prompt(tok, name, ex, MAX_LEN).cuda(), window=WINDOW)
    backup = cgr.make_backup(pf)
    print(f"[{name}] {pf.n} prompt tokens, prefill {pf.seconds:.1f}s | gold: {ex['answers'][:2]}")
    for cfg in (cgr.GenConfig(method="full"), cgr.GenConfig(method="snapkv", ratio=0.05),
                cgr.GenConfig(method="snapkv", ratio=0.05, trigger="conf", tau=HEAD_TAU)):
        cfg.max_new_tokens = cgr.MAX_NEW[name]
        r = cgr.generate(model, tap, pf, cfg, EOS, backup)
        text = tok.decode(r.tokens, skip_special_tokens=True).strip()
        print(f"   {cfg.name():22s} score {cgr.score_output(name, text, ex['answers']):.2f} | {len(r.trig_steps)} recoveries | {text[:80]!r}")
        if cfg.method == "full":
            assert any(ch.isalnum() for ch in text), "the full-cache answer is not text; do not run the experiments"
    del pf, backup; gc.collect(); torch.cuda.empty_cache()
print("pilot OK")

In [ ]:
grid = cgr.config_grid(RATIOS, TAUS)
print(len(grid), "configurations per example,", sum(len(v) for v in data.values()), "examples")
t0 = time.time()
for name in DATASETS:
    cgr.run_dataset(model, tok, tap, name, data[name], RESULTS, RATIOS, TAUS,
                    max_len=MAX_LEN, window=WINDOW, ref_tau=HEAD_TAU)
print(f"done in {(time.time() - t0) / 60:.1f} min")

## 8. Results

Everything below is computed from `results.jsonl` only. Only examples for which every configuration finished
are used, so all comparisons are paired.

In [ ]:
import numpy as np, pandas as pd
pd.set_option("display.width", 200)

df = pd.read_json(open(RESULTS, encoding="utf-8"), lines=True)
per_ex = df.groupby(["dataset", "idx"]).name.count()
complete = per_ex[per_ex == per_ex.max()].index
df = df.set_index(["dataset", "idx"]).loc[complete].reset_index()
print("complete examples per dataset:", df[df.name == "full"].groupby("dataset").idx.count().to_dict())

HEAD = f"snapkv+cgr(tau={HEAD_TAU})"
df["rate"] = df.n_trig / df.steps
full_pred = df[df.name == "full"].set_index(["dataset", "idx"]).pred
# fidelity: token-F1 between a method's answer and the uncompressed model's answer on the same example
df["fidelity"] = [cgr.qa_f1(p, [full_pred[(d, i)]]) if full_pred[(d, i)].strip() else float(not p.strip())
                  for p, d, i in zip(df.pred, df.dataset, df.idx)]
df["ms_per_token"] = 1000 * df.gen_seconds / df.steps

LABEL = {"full": "Full cache", "random": "Random", "streaming": "StreamingLLM", "h2o": "H2O", "snapkv": "SnapKV",
         HEAD: "SnapKV + CGR (ours)", "snapkv+randtrig": "SnapKV + random trigger", "snapkv+always": "SnapKV + always recover"}
main = df[df.name.isin(LABEL)].copy()
main["Method"] = main.name.map(LABEL)
order = list(LABEL.values())

def table(value):
    t = main[main.name != "full"].pivot_table(index="Method", columns=["dataset", "ratio"], values=value, aggfunc="mean") * 100
    avg = main[main.name != "full"].pivot_table(index="Method", columns="ratio", values=value, aggfunc="mean") * 100
    avg.columns = pd.MultiIndex.from_product([["AVERAGE"], avg.columns])
    return pd.concat([t, avg], axis=1).reindex([m for m in order if m in t.index]).round(1)

t1 = table("score")
print("\nTABLE 1. Task score x100 (F1 for QA, recall for needle). Columns: dataset / fraction of context kept.")
print(t1.to_string())
print("\nFull cache:", (main[main.name == "full"].groupby("dataset").score.mean() * 100).round(1).to_dict(),
      "| average", round(main[main.name == "full"].score.mean() * 100, 1))
t1b = table("fidelity")
print("\nTABLE 1b. Agreement with the uncompressed model's answer x100 (token F1).")
print(t1b.to_string())
t1.to_csv(f"{OUT_DIR}/table1_score.csv"); t1b.to_csv(f"{OUT_DIR}/table1b_fidelity.csv")

In [ ]:
# TABLE 2. What recovery costs and what it does, pooled over datasets.
rec = df[df.trigger != "never"].copy()
rec["Variant"] = rec.name.map(lambda n: LABEL.get(n, n))
conf_gain = rec.apply(lambda r: np.mean(r.conf_after) - np.mean([r.conf[s] for s in r.trig_steps]) if r.n_trig else np.nan, axis=1)
rec["conf_gain"] = conf_gain
t2 = rec.groupby(["ratio", "name"]).agg(
    score=("score", lambda s: 100 * s.mean()), fidelity=("fidelity", lambda s: 100 * s.mean()),
    trigger_rate_pct=("rate", lambda s: 100 * s.mean()),
    examples_triggered_pct=("n_trig", lambda s: 100 * (s > 0).mean()),
    token_changed_pct=("flipped", "sum"), triggers=("n_trig", "sum"),
    slots_swapped_pct=("swapped", lambda s: 100 * s[s > 0].mean()),
    conf_gain=("conf_gain", "mean"), recover_ms=("rec_seconds", "sum")).reset_index()
t2["token_changed_pct"] = 100 * t2.token_changed_pct / t2.triggers.clip(lower=1)
t2["recover_ms"] = 1000 * t2.recover_ms / t2.triggers.clip(lower=1)
snap = df[df.name == "snapkv"].groupby("ratio").agg(score=("score", lambda s: 100 * s.mean()),
                                                     fidelity=("fidelity", lambda s: 100 * s.mean())).reset_index()
snap["name"] = "snapkv"
t2 = pd.concat([snap, t2]).sort_values(["ratio", "trigger_rate_pct"], na_position="first").round(2)
print("TABLE 2. Recovery behaviour (all datasets pooled). recover_ms = wall time of one recovery incl. the re-decoded step.")
print(t2.to_string(index=False))
t2.to_csv(f"{OUT_DIR}/table2_recovery.csv", index=False)

# Paired bootstrap: is the gain over SnapKV, and over random triggering, more than noise?
def paired_ci(a, b, n_boot=10000, seed=0):
    d = (a - b).dropna().to_numpy()
    rng = np.random.default_rng(seed)
    boots = rng.choice(d, (n_boot, len(d))).mean(1)
    return 100 * d.mean(), 100 * np.percentile(boots, 2.5), 100 * np.percentile(boots, 97.5), len(d)

wide = df.pivot_table(index=["dataset", "idx", "ratio"], columns="name", values="score")
rows = []
for ratio in RATIOS:
    w = wide.xs(ratio, level="ratio")
    for other in ("snapkv", "snapkv+randtrig", "snapkv+always"):
        if other in w and HEAD in w:
            m, lo, hi, n = paired_ci(w[HEAD], w[other])
            rows.append({"ratio": ratio, "comparison": f"ours - {LABEL[other]}", "mean_diff": m, "ci_low": lo, "ci_high": hi, "n": n})
t3 = pd.DataFrame(rows).round(2)
print("\nTABLE 3. Paired differences in score x100 with 95% bootstrap intervals (an interval that excludes 0 is significant).")
print(t3.to_string(index=False))
t3.to_csv(f"{OUT_DIR}/table3_significance.csv", index=False)

# TABLE 4. Memory and speed. KV sizes are counted from the tensors, not from torch.cuda.max_memory_allocated.
mem = df[df.name.isin(["full", "snapkv", HEAD])].groupby(["name", "ratio"]).agg(
    gpu_kv_mb=("gpu_kv_bytes", lambda s: s.mean() / 2**20), backup_cpu_mb=("backup_bytes", lambda s: s.mean() / 2**20),
    kept_tokens=("kept_tokens", "mean"), prompt_tokens=("n_prompt", "mean"),
    ms_per_token=("ms_per_token", "mean"), prefill_s=("prefill_seconds", "mean")).reset_index()
mem["gpu_kv_saving_pct"] = 100 * (1 - mem.gpu_kv_mb / mem[mem.name == "full"].gpu_kv_mb.iloc[0])
print("\nTABLE 4. Memory and speed (means over all examples). backup_cpu_mb is the CPU RAM our method adds.")
print(mem.round(2).to_string(index=False))
mem.round(3).to_csv(f"{OUT_DIR}/table4_memory_speed.csv", index=False)

In [ ]:
import matplotlib as mpl, matplotlib.pyplot as plt
mpl.rcParams.update({"font.size": 8, "axes.spines.top": False, "axes.spines.right": False, "axes.grid": True,
                     "grid.color": "#e6e6e3", "grid.linewidth": 0.6, "axes.edgecolor": "#52514e",
                     "axes.labelcolor": "#0b0b0b", "xtick.color": "#52514e", "ytick.color": "#52514e",
                     "legend.frameon": False, "savefig.bbox": "tight", "savefig.dpi": 300, "pdf.fonttype": 42})
# One fixed colour, marker and line style per method, so the figures also read in greyscale print.
STYLE = {"StreamingLLM": ("#eda100", "v", ":"), "H2O": ("#1baf7a", "s", "-."), "SnapKV": ("#2a78d6", "o", "--"),
         "SnapKV + CGR (ours)": ("#eb6834", "D", "-"), "SnapKV + random trigger": ("#6250d6", "^", ":"),
         "SnapKV + always recover": ("#52514e", "P", "-.")}
pct = [100 * r for r in RATIOS]

def save(fig, name):
    fig.savefig(f"{OUT_DIR}/{name}.pdf"); fig.savefig(f"{OUT_DIR}/{name}.png"); plt.show()

# FIGURE 1. Score against budget, one panel per dataset.
panels = DATASETS + ["average"]
fig, axes = plt.subplots(1, len(panels), figsize=(2.05 * len(panels), 2.3), sharex=True)
for ax, name in zip(axes, panels):
    sub = main if name == "average" else main[main.dataset == name]
    ax.axhline(100 * sub[sub.name == "full"].score.mean(), color="#0b0b0b", lw=1, ls=(0, (1, 1)), label="Full cache")
    for m in ("StreamingLLM", "H2O", "SnapKV", "SnapKV + CGR (ours)"):
        y = 100 * sub[sub.Method == m].groupby("ratio").score.mean().reindex(RATIOS)
        c, mk, ls = STYLE[m]
        ax.plot(pct, y, color=c, marker=mk, ls=ls, lw=1.6, ms=4.5, label=m)
    ax.set_xscale("log"); ax.set_xticks(pct); ax.set_xticklabels([f"{p:g}" for p in pct]); ax.minorticks_off()
    ax.set_title(name, fontsize=8.5); ax.set_xlabel("context kept (%)")
axes[0].set_ylabel("score (x100)")
h, l = axes[0].get_legend_handles_labels()
fig.legend(h, l, loc="upper center", ncol=5, bbox_to_anchor=(0.5, 1.08), fontsize=7.5)
fig.tight_layout(); save(fig, "fig1_score_vs_budget")

# FIGURE 2. Score against how often recovery fires, one panel per budget, all datasets pooled.
fig, axes = plt.subplots(1, len(RATIOS), figsize=(2.05 * len(RATIOS), 2.3), sharey=True)
for ax, ratio in zip(np.atleast_1d(axes), RATIOS):
    sub = df[df.ratio == ratio]
    ax.axhline(100 * df[df.name == "full"].score.mean(), color="#0b0b0b", lw=1, ls=(0, (1, 1)), label="Full cache")
    pts = sub[sub.trigger == "conf"].groupby("tau").agg(x=("rate", "mean"), y=("score", "mean")).sort_index()
    s0 = sub[sub.name == "snapkv"].score.mean()
    c, mk, ls = STYLE["SnapKV + CGR (ours)"]
    ax.plot([0] + list(100 * pts.x), [100 * s0] + list(100 * pts.y), color=c, ls=ls, lw=1.6, marker=mk, ms=4.5, label="SnapKV + CGR (ours)")
    for k, (tau, r) in enumerate(pts.iterrows()):   # labels staggered downwards so close points stay readable
        ax.annotate(f"τ={tau:g}", (100 * r.x, 100 * r.y), textcoords="offset points", xytext=(5, -10 - 8 * k), fontsize=6.5,
                    color="#52514e", arrowprops={"arrowstyle": "-", "color": "#b5b4ae", "lw": 0.5})
    for nm in ("snapkv", "snapkv+randtrig", "snapkv+always"):
        g = sub[sub.name == nm]
        if len(g):
            c, mk, _ = STYLE[LABEL[nm]]
            ax.plot([100 * g.rate.mean()], [100 * g.score.mean()], color=c, marker=mk, ms=6, ls="none",
                    markeredgecolor="white", markeredgewidth=0.8, label=LABEL[nm], zorder=3)
    ax.set_xscale("symlog", linthresh=5); ax.set_xlim(-0.5, 130); ax.set_xticks([0, 5, 20, 100]); ax.set_xticklabels(["0", "5", "20", "100"])
    ax.set_title(f"{100 * ratio:g}% of context kept", fontsize=8.5); ax.set_xlabel("steps with recovery (%)")
np.atleast_1d(axes)[0].set_ylabel("score (x100)")
h, l = np.atleast_1d(axes)[0].get_legend_handles_labels()
fig.legend(h, l, loc="upper center", ncol=3, bbox_to_anchor=(0.5, 1.16), fontsize=7.5)
fig.tight_layout(); save(fig, "fig2_score_vs_trigger_rate")

# FIGURE 3. Confidence at the steps where recovery fired, before and after (ours, all budgets).
ours = df[(df.name == HEAD) & (df.n_trig > 0)]
before = np.concatenate([[r.conf[s] for s in r.trig_steps] for r in ours.itertuples()]) if len(ours) else np.array([])
after = np.concatenate([r.conf_after for r in ours.itertuples()]) if len(ours) else np.array([])
fig, ax = plt.subplots(figsize=(3.4, 2.2))
bins = np.linspace(0, 1, 21)
ax.hist(before, bins, color="#2a78d6", alpha=0.85, label=f"before recovery (mean {before.mean():.2f})" if len(before) else "before")
ax.hist(after, bins, histtype="step", color="#eb6834", lw=1.8, label=f"after recovery (mean {after.mean():.2f})" if len(after) else "after")
ax.axvline(HEAD_TAU, color="#52514e", lw=0.8, ls="--"); ax.text(HEAD_TAU + 0.01, ax.get_ylim()[1] * 0.92, "τ", fontsize=8, color="#52514e")
ax.set_xlabel("next-token confidence at trigger steps"); ax.set_ylabel("number of steps"); ax.legend(fontsize=7)
fig.tight_layout(); save(fig, "fig3_confidence_before_after")
print(f"{len(before)} recoveries in total")

## 9. Examples and download

In [ ]:
# A few examples where recovery changed the answer, at the smallest budget. Useful for the paper's qualitative section.
r0 = min(RATIOS)
a = df[(df.name == "snapkv") & (df.ratio == r0)].set_index(["dataset", "idx"])
b = df[(df.name == HEAD) & (df.ratio == r0)].set_index(["dataset", "idx"])
changed = [k for k in a.index if a.pred[k] != b.pred[k]]
print(f"budget {100 * r0:g}%: recovery changed the answer on {len(changed)} of {len(a)} examples; "
      f"better on {sum(b.score[k] > a.score[k] for k in changed)}, worse on {sum(b.score[k] < a.score[k] for k in changed)}")
for k in sorted(changed, key=lambda k: a.score[k] - b.score[k])[:6]:
    print(f"\n[{k[0]} #{k[1]}]\n  full   : {full_pred[k][:90]!r}\n  snapkv : {a.pred[k][:90]!r}  (score {a.score[k]:.2f})"
          f"\n  ours   : {b.pred[k][:90]!r}  (score {b.score[k]:.2f}, {b.n_trig[k]} recoveries)")

import shutil
shutil.make_archive(f"{BASE}/cgr_outputs", "zip", OUT_DIR)
try:
    from google.colab import files
    files.download(f"{BASE}/cgr_outputs.zip")
except Exception as e:
    print(f"zip is at {BASE}/cgr_outputs.zip", e)